# 03 · Fetch stress test

The wrapper checklist from
[`TESTING_AND_IMPROVEMENTS.md`](../../TESTING_AND_IMPROVEMENTS.md): monthly
period parsing, `code_list=` on a variable other than `Alder`, real batching
above SSB's 800k-cell limit, `language="en"`, and `.to_polars()`.

**This notebook also contains the most serious finding of the whole exercise**
(section 5) — a crash, not a wrinkle.

In [1]:
import time
import warnings

import pandas as pd

from ssb_statistikkbanken import SSB

pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 45)

ssb = SSB()

## 1 · Monthly table → the `date` column

`_derive_date()` had **never** run against a real table before this — every
fixture and every earlier notebook used annual data. Table `08329` is monthly
(`1999M01` … `2026M06`).

In [2]:
monthly = ssb.fetch("08329", Skadegrad="01", latest=6)
monthly[["Tid", "Tid_code", "date", "value"]]

,Tid,Tid_code,date,value
0,2026M01,2026M01,2026-01-01,11
1,2026M02,2026M02,2026-02-01,4
2,2026M03,2026M03,2026-03-01,6
3,2026M04,2026M04,2026-04-01,6
4,2026M05,2026M05,2026-05-01,15
5,2026M06,2026M06,2026-06-01,10


In [3]:
print(monthly.dtypes.to_string())
print()
print(f"'date' derived: {'date' in monthly.columns}")
print(f"'year' absent (correct for monthly): {'year' not in monthly.columns}")

Skadegrad                       str
Skadegrad_code                  str
ContentsCode                    str
ContentsCode_code               str
Tid                             str
Tid_code                        str
date                 datetime64[us]
value                         Int64

'date' derived: True
'year' absent (correct for monthly): True


`2026M01` → `2026-01-01` as a real `datetime64`. The month code is parsed to the
first day of the period, and `year` is correctly **not** emitted. This path works.

## 2 · `code_list=` on `Region`, not `Alder`

Every previous test of code lists used `Alder`. Table `14346` exposes
`vs_Landet` and `vs_Fylker2024` on `Region`.

In [4]:
by_fylke = ssb.fetch(
    "14346",
    code_list={"Region": "vs_Fylker2024"},
    AktsomhetsOmr="01_flom",
    BygnType="2",
    latest=1,
)
print(f"{len(by_fylke)} rows -- the code list grouped Region into counties")
by_fylke[["Region", "Region_code", "AktsomhetsOmr", "value"]].head()

16 rows -- the code list grouped Region into counties


,Region,Region_code,AktsomhetsOmr,value
0,Østfold,31,Flom,2284
1,Akershus,32,Flom,3211
2,Oslo - Oslove,03,Flom,539
3,Innlandet,34,Flom,20579
4,Buskerud,33,Flom,7025


## 3 · `language="en"`

Values translate; **column names do not move**, because `columns="code"` is the
default. That is the whole point of that default — an English run does not break
a script written against Norwegian output.

In [5]:
english = ssb.fetch(
    "14346",
    code_list={"Region": "vs_Fylker2024"},
    AktsomhetsOmr="01_flom",
    BygnType="2",
    latest=1,
    language="en",
)
print(f"columns identical across languages: {list(by_fylke.columns) == list(english.columns)}")
english[["Region", "Region_code", "AktsomhetsOmr", "ContentsCode"]].head(3)

columns identical across languages: True


,Region,Region_code,AktsomhetsOmr,ContentsCode
0,Østfold,31,Floods,Buildings
1,Akershus,32,Floods,Buildings
2,Oslo - Oslove,03,Floods,Buildings


## 4 · Real batching above 800,000 cells

SSB rejects any single request over 800k cells. pxwebpy is supposed to split the
selection, fetch the parts, and stitch them back. That had never been verified
against the live API.

Selection below is **898,560 cells** — deliberately just over the limit.

In [6]:
info = ssb.describe("08329")
months = info["Tid"].codes[-120:]

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    start = time.time()
    big = ssb.fetch(
        "08329",
        Alder2="*",
        Skadegrad="*",
        VegtrTrafGrp="*",
        Ulukkesgruppe="*",
        time=months,
    )
    elapsed = time.time() - start

print(f"rows returned : {len(big):,}")
print(f"expected cells: {13 * 6 * 8 * 12 * 120:,}")
print(f"exact match   : {len(big) == 13 * 6 * 8 * 12 * 120}")
print(f"elapsed       : {elapsed:.1f}s")
print()
for warning in caught:
    print(f"warning: {warning.message}")

rows returned : 898,560
expected cells: 898,560
exact match   : True
elapsed       : 6.4s



Batching works, and the warning tells you *before* the request how many pieces
it will be split into. ~900k rows in a handful of seconds.

## 5 · Tables that mix units — the bug this exercise found

Table `13902` (naturfare, kommune level) reports **`decimals = 0`** at the table
level, but its two statistics have different units:

- `Bygninger` — building counts, integers
- `Veger (km)` — road length, decimals like `0.2`, `0.7`

`decimals` is a **table-level** field, so the wrapper used to cast the whole
`value` column to `Int64`. That made **pandas raise `TypeError`** (data
unreachable) and **polars silently truncate `0.2` to `0`** (data corrupted —
arguably the worse of the two).

**Fixed (F-05):** `decimals` is now a hint, and the actual values get the final
say. Both backends decide together, so they can never disagree.

In [7]:
dataset = ssb.dataset("13902", Region="0301", latest=1)
print(f"spec.integer_value    = {dataset.spec.integer_value}   <- the table-level hint")
print(f"_use_integer_values() = {dataset._use_integer_values()}   <- what the data says")
print()
for record in dataset.to_records()[:6]:
    value = record["value"]
    print(f"   {record['ContentsCode_code']:<10} {value!r:<6} ({type(value).__name__})")

spec.integer_value    = True   <- the table-level hint
_use_integer_values() = False   <- what the data says

   Bygninger  2      (int)
   Veger      0.2    (float)
   Bygninger  11     (int)
   Veger      0.7    (float)
   Bygninger  0      (int)
   Veger      0      (int)


In [8]:
frame = dataset.to_pandas()
print(f"to_pandas() OK -- dtype={frame['value'].dtype}")
print(f"0.2 preserved: {0.2 in frame['value'].tolist()}")
frame[["ContentsCode_code", "value"]].head()

to_pandas() OK -- dtype=Float64
0.2 preserved: True


,ContentsCode_code,value
0,Bygninger,2.0
1,Veger,0.2
2,Bygninger,11.0
3,Veger,0.7
4,Bygninger,0.0


### Blast radius

All four naturfare tables now work, and each still gets the dtype its data
actually warrants — the fix did not flatten everything to float:

In [9]:
survey = []
for table_id in ["13902", "14345", "14346", "14347"]:
    meta = ssb.describe(table_id)
    try:
        frame = ssb.fetch(table_id, latest=1)
        outcome = f"OK ({frame['value'].dtype})"
    except TypeError:
        outcome = "CRASH"
    survey.append(
        {
            "table": table_id,
            "decimals": meta.decimals,
            "contents": ", ".join(meta["ContentsCode"].values.values()),
            "fetch()": outcome,
        }
    )
pd.DataFrame(survey)

,table,decimals,contents,fetch()
0,13902,0,"Bygninger, Veger (km)",OK (Float64)
1,14345,0,"Bygninger, Veger (km)",OK (Float64)
2,14346,0,Bygninger,OK (Int64)
3,14347,1,Veger (km),OK (Float64)


`14346` (counts only) keeps `Int64`; `14347` (km only) keeps `Float64`; the two
mixed tables now come back as `Float64` instead of crashing.

### Per-statistic selections are typed on their own merits

In [10]:
for contents in ["Bygninger", "Veger"]:
    frame = ssb.fetch("13902", Region="0301", contents=contents, latest=1)
    print(f"contents={contents:<10} dtype={frame['value'].dtype}")

contents=Bygninger  dtype=Int64


contents=Veger      dtype=Float64


Selecting only `Bygninger` keeps `Int64` even though it comes from a table whose
sibling statistic is decimal — the decision follows the data that was actually
returned, not the table-wide hint.

### The backends agree

The old behaviour diverged silently: pandas raised while polars truncated. Now
both look at the same values and reach the same answer.

In [11]:
from ssb_statistikkbanken import MissingDependencyError

try:
    polars_frame = dataset.to_polars()
    print(f"polars value dtype : {polars_frame.schema['value']}")
    print(f"pandas value dtype : {frame['value'].dtype}")
    print(f"0.2 preserved      : {0.2 in polars_frame['value'].to_list()}")
except MissingDependencyError:
    print("polars not installed in the kernel -- see F-06 in TESTING_AND_IMPROVEMENTS.md")

polars not installed in the kernel -- see F-06 in TESTING_AND_IMPROVEMENTS.md


## 6 · `.to_polars()` with polars actually installed

Only the *missing-dependency* path had ever been tested.

⚠️ **F-06:** running `uv run --with polars jupyter …` does **not** give the
notebook kernel polars — the kernelspec registered by `make kernel` hardcodes
`.venv/bin/python3`. Anything a notebook needs has to be a real project
dependency. So this cell may well show the fallback.

In [12]:
from ssb_statistikkbanken import MissingDependencyError

small = ssb.dataset("08329", Skadegrad="01", latest=4)
try:
    polars_frame = small.to_polars()
    print(polars_frame)
    print()
    print("schema:", dict(polars_frame.schema))
except MissingDependencyError as exc:
    print(f"polars not installed -- error path instead:\n{exc}")

polars not installed -- error path instead:
polars is required for .to_polars(). Install it with:
  uv add polars
or use .to_pandas() / .to_records() instead.
